In [1]:
# Transformer의 Self-Attention 이해를 위해 간단한 숫자 데이터로 실습.
# 3개의 토큰("A", "B", "C")을 2차원 임베딩으로 표현하고, Self-Attention(Q,K,V 동일한 입력) 연산 과정을 단계별로 보여 준다.

import numpy as np

# 1) 입력 임베딩 준비 (3 tokens × 2-dim)
#    예: A=[1,0], B=[0,1], C=[1,1]
X = np.array( [     # 입력 임베딩 X,  각 토큰을 2차원 벡터로 표현 (A=[1,0], B=[0,1], C=[1,1]).
    [1.0, 0.0],    # A
    [0.0, 1.0],    # B
    [1.0, 1.0],    # C
] )  # shape (3,2)

# 2) Query, Key, Value 생성 (Self-Attention 이므로 동일)
#    Self-Attention에서는 모두 같은 X를 사용.   실제론 Wq, Wk, Wv를 학습하지만, 여기선 단순화 위해 X 그대로 사용
Q = X.copy()
K = X.copy()
V = X.copy()

# 3) Attention score 계산: Q•Kᵀ / √d   각 토큰 𝑖가 토큰 𝑗와 얼마나“유사한지”를 수치화.
d_k = Q.shape[1]     # 2
scores = Q.dot(K.T) / np.sqrt(d_k)
  # scores shape = (3,3):
  #   [
  #     [ (1*1+0*0)/√2, (1*0+0*1)/√2, (1*1+0*1)/√2 ],
  #     [ ... ],
  #     [ ... ]
  #   ]

# 4) Softmax로 가중치(normalized attention)
# 각 행(각 Query 𝑖)에 대해 exp(score)를 합이 1이 되도록 정규화 → attention weight.
def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

weights = softmax(scores)   # weights shape = (3,3): 각 토큰이 모든 토큰에 대해 집중하는 정도

# 5) 최종 출력: weights × V        (3,3) × (3,2) → (3,2)
outputs = weights.dot(V)   # 토큰 𝑖가 어느 토큰에 집중했는지에 따라 Value를 가중합해 새로운 표현을 만듦.

print("입력 임베딩 X:\n", X, "\n")
print("Attention scores (Q•Kᵀ/√d):\n", np.round(scores,3), "\n")
print("Attention weights (softmax):\n", np.round(weights,3), "\n")
print("Self-Attention 출력 (weights•V):\n", np.round(outputs,3))

# 이렇게 하면, 입력된 세 토큰 각각이 자기 자신과 다른 토큰을 얼마나 참조(attend)했는지 확인해 볼 수 있다.  간단하지만 Self-Attention의 핵심 아이디어:
#   '내가 가진 Query(나 자신)를 가지고, 모든 Key(나를 포함한 모든 단어)와 비교해
#     attention weight를 구하고, 이를 이용해 Value를 가중합한다.'
# 를 잘 보여 준다.

입력 임베딩 X:
 [[1. 0.]
 [0. 1.]
 [1. 1.]] 

Attention scores (Q•Kᵀ/√d):
 [[0.707 0.    0.707]
 [0.    0.707 0.707]
 [0.707 0.707 1.414]] 

Attention weights (softmax):
 [[0.401 0.198 0.401]
 [0.198 0.401 0.401]
 [0.248 0.248 0.503]] 

Self-Attention 출력 (weights•V):
 [[0.802 0.599]
 [0.599 0.802]
 [0.752 0.752]]
